# QNN — Iris Dataset


## Librerias


In [ ]:
#1.Import all libraries
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, label_binarize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


## Data


In [ ]:
data = pd.DataFrame(datasets.load_iris().data, columns=datasets.load_iris().feature_names)
data['target'] = datasets.load_iris().target
data


In [ ]:
data.target.unique()


## Data Preparation


In [ ]:
#2. Dataset preparation
torch.manual_seed(123)
np.random.seed(123)

n_qubits = 4
n_classes = 3

X = data.drop(columns='target').to_numpy()
y = data['target'].to_numpy()

X_train_raw, X_test_raw, y_train_np, y_test_np = train_test_split(
    X, y, test_size=0.4, random_state=123
)

# Normalize to [0, π] for Angle Embedding
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

X_train = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.long)
y_test = torch.tensor(y_test_np, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"Custom Dataset Loaded: Train={len(X_train)}, Test={len(X_test)}")
print(f"Final shapes -> Train: {tuple(X_train.shape)}, Test: {tuple(X_test.shape)}")


## Quantum Neural Network


In [ ]:
#3. QNN model definition
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    # Angle encoding: map 4 features to 4 qubits
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    # Entanglement layer
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

n_layers = 6
weight_shapes = {"weights": (n_layers, n_qubits)}

class QNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)
        self.post = nn.Linear(n_qubits, n_classes)

    def forward(self, x):
        x = self.qlayer(x)
        return self.post(x)


## Entrenamiento QNN


In [ ]:
#5.Generic training function
def train_model(model, loader, epochs=30, lr=0.01):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = []
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for bx, by in loader:
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(loader)
        history.append(avg_loss)
        print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f}")
    return history


In [ ]:
#6.Train QNN + timing
qnn_model = QNN()
start = time.time()
qnn_history = train_model(qnn_model, train_loader, epochs=30, lr=0.01)
qnn_time = time.time() - start
print(f"QNN training time: {qnn_time:.2f}s")


## Model Evaluation


In [ ]:
#8.Evaluation function + print results
@torch.no_grad()
def evaluate(model, X_t, y_t, classes):
    model.eval()
    logits = model(X_t)
    preds = torch.argmax(logits, dim=1).numpy()
    probs = nn.functional.softmax(logits, dim=1).numpy()

    y_true = y_t.numpy()
    acc = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, average="macro", zero_division=0)
    rec = recall_score(y_true, preds, average="macro", zero_division=0)
    f1 = f1_score(y_true, preds, average="macro", zero_division=0)
    auc = roc_auc_score(label_binarize(y_true, classes=classes), probs, average="macro")
    return acc, prec, rec, f1, auc, probs, preds

classes = np.unique(y_train_np)
qnn_acc, qnn_prec, qnn_rec, qnn_f1, qnn_auc, qnn_probs, qnn_preds = evaluate(
    qnn_model, X_test, y_test, classes
)

print("\n=== Results ===")
print(
    f"QNN - Acc: {qnn_acc:.4f} | Prec: {qnn_prec:.4f} | Rec: {qnn_rec:.4f} | "
    f"F1: {qnn_f1:.4f} | AUC: {qnn_auc:.4f}"
)


In [ ]:
#9.Result Visualization (multiclase + paleta UNIR)
plt.rcParams.update({
    'font.sans-serif': ['Arial'],
    'axes.edgecolor': '#C0C0C8',
    'grid.color': '#C0C0C8',
    'grid.linestyle': '--'
})

@torch.no_grad()
def get_inference_time(model, data):
    start = time.time()
    _ = model(data)
    return time.time() - start

qnn_inf_time = get_inference_time(qnn_model, X_test)

quantum_color = '#219BC3'  # unirazul
unir_gris = '#858591'

fig, axs = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
fig.suptitle('QNN Performance Analysis', fontsize=18, y=1.02, color='#0E4A75')

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
qnn_vals = [qnn_acc, qnn_prec, qnn_rec, qnn_f1, qnn_auc]
x = np.arange(len(metrics))
width = 0.5

for i in range(len(metrics)):
    axs[0].bar(x[i], qnn_vals[i], width, color=quantum_color,
               label='QNN' if i == 0 else '', zorder=3)
    axs[0].text(x[i], qnn_vals[i] + 0.02, f'{qnn_vals[i]:.4f}',
                ha='center', va='bottom', fontsize=9, color=unir_gris)

axs[0].set_ylabel('Score')
axs[0].set_title('Evaluation Metrics', color='#0E4A75')
axs[0].set_xticks(x)
axs[0].set_xticklabels(metrics)
axs[0].set_ylim(0, 1.35)
axs[0].legend(loc='upper right')
axs[0].grid(axis='y', zorder=0)

eff_labels = ['Training Time', 'Inference Time']
qnn_eff = [qnn_time, qnn_inf_time]
x_eff = np.arange(len(eff_labels))

axs[1].bar(x_eff, qnn_eff, width, color=quantum_color, label='QNN', zorder=3)
axs[1].set_xticks(x_eff)
axs[1].set_xticklabels(eff_labels)
axs[1].set_ylabel('Time (s)')
axs[1].set_title('Computational Efficiency', color='#0E4A75')
max_time = max(qnn_eff)
axs[1].set_ylim(0, max_time * 1.35 if max_time > 0 else 1)
axs[1].legend(loc='upper right')
axs[1].grid(axis='y', zorder=0)

offset = max_time * 0.02 if max_time > 0 else 0.02
for i in range(2):
    axs[1].text(x_eff[i], qnn_eff[i] + offset, f'{qnn_eff[i]:.4f}s',
                ha='center', va='bottom', fontsize=9, color=unir_gris)

plt.tight_layout()
plt.show()
